# Lab 3 – Evaluate AI Agents with Azure AI Evaluation SDK

## Scenario
In this lab, you will evaluate the quality, safety, and task adherence of the agents you created in **Lab 1** and **Lab 2**.

You will use the **Azure AI Evaluation SDK** to run automated evaluations against agent responses using built-in evaluators such as **Relevance**, **Groundedness**, and **Task Adherence**.

This lab demonstrates **offline, repeatable evaluation**—a critical step before deploying agents into production.

## Learning Objectives
By the end of this lab, you will be able to:
- Install and configure the Azure AI Evaluation SDK
- Create an evaluation dataset for an AI agent
- Run quality and agentic evaluators
- Interpret evaluation metrics
- Understand how evaluation fits into CI/CD and GenAIOps workflows

## Prerequisites
- Completed **Lab 1** and **Lab 2**
- Python 3.10+
- Azure CLI authenticated (`az login`)
- AI Foundry Project Endpoint
- ProductInventoryManager Agent ID

## Step 1 — Install evaluation dependencies
Run in your active Python environment.

In [4]:

%pip install --pre azure-ai-evaluation
%pip install azure-identity python-dotenv

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 7.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.5/434.5 kB 7.0 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 13.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 15.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.3/197.3 kB 12.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.0/218.0 kB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 9.2 MB/s e

## Step 2 — Configure environment variables
Ensure the following variables are set (reuse from previous labs).

In [9]:
# .env file
AZURE_FOUNDRY_PROJECT_ENDPOINT="https://lapate-5938-resource.services.ai.azure.com/api/projects/lapate-5938"
PRODUCT_INVENTORY_MANAGER_AGENT_ID="ProductInventoryManager"
AZURE_FOUNDRY_PROJECT_MODEL_ID="gpt-4o-mini"

## Step 3 — Load configuration and authenticate

In [11]:
%pip install python-dotenv
%pip install azure-ai-projects

import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

endpoint = os.environ["AZURE_FOUNDRY_PROJECT_ENDPOINT"]
agent_id = os.environ["PRODUCT_INVENTORY_MANAGER_AGENT_ID"]
model_id = os.getenv("AZURE_FOUNDRY_PROJECT_MODEL_ID", "gpt-4o-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

print("Connected to Foundry project")

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Connected to Foundry project


## Step 4 — Create a small evaluation dataset
We will evaluate the agent on a small, controlled set of prompts.

The dataset uses **query / response / ground_truth** format.

In [13]:
import json
from pathlib import Path

samples = [
    {
        "query": "Describe our top-selling product.",
        "response": "The Alpine Explorer Tent is our top-selling product.",
        "ground_truth": "The Alpine Explorer Tent is the most popular product based on sales."
    },
    {
        "query": "Which product is most popular?",
        "response": "The Alpine Explorer Tent leads sales.",
        "ground_truth": "The Alpine Explorer Tent is the most popular product."
    }
]

path = Path("eval_data.jsonl")
with path.open("w") as f:
    for row in samples:
        f.write(json.dumps(row) + "")

path

PosixPath('eval_data.jsonl')

## Step 5 — Initialize evaluators
We will use built-in evaluators from the Azure AI Evaluation SDK.

In [31]:
from azure.ai.evaluation import (
    RelevanceEvaluator,
    GroundednessEvaluator,
    QAEvaluator,
    AzureOpenAIModelConfiguration
)
from azure.identity import get_bearer_token_provider

token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default"
)

model_config: AzureOpenAIModelConfiguration = {
    "type": "azure_openai",
    "azure_endpoint": endpoint,
    "azure_deployment": model_id,
    "api_version": "2024-06-01"
}

relevance = RelevanceEvaluator(model_config=model_config)
groundedness = GroundednessEvaluator(model_config=model_config)
qa = QAEvaluator(model_config=model_config)

## Step 6 — Run the evaluation

In [32]:
from azure.ai.evaluation import evaluate

# Ensure valid JSONL: one JSON object per line
with path.open("w", encoding="utf-8") as f:
    for row in samples:
        f.write(json.dumps(row) + "\n")

results = evaluate(
    data=str(path),
    evaluators={
        "relevance": relevance,
        "groundedness": groundedness,
        "qa": qa
    },
    output_path="eval_results.json"
)

results

{'rows': [{'inputs.query': 'Describe our top-selling product.',
   'inputs.response': 'The Alpine Explorer Tent is our top-selling product.',
   'inputs.ground_truth': 'The Alpine Explorer Tent is the most popular product based on sales.'},
  {'inputs.query': 'Which product is most popular?',
   'inputs.response': 'The Alpine Explorer Tent leads sales.',
   'inputs.ground_truth': 'The Alpine Explorer Tent is the most popular product.'}],
 'metrics': {},
 'studio_url': None}

## Step 7 — Review evaluation metrics

In [30]:
print("Metrics:")
print(results["metrics"])

## Step 8 — Interpret the results
- **Relevance**: Is the response related to the query?
- **Groundedness**: Is the response supported by known facts?
- **QA**: Composite score across correctness dimensions

These metrics help establish a **baseline** before deploying agents.

## Step 9 — Where this fits in production
- Run locally during development
- Integrate into CI/CD pipelines
- Track regressions across agent versions
- Enforce quality gates before deployment

## Validation Checklist
- ✅ Evaluation dataset created
- ✅ Evaluators initialized
- ✅ Metrics generated successfully
- ✅ Results interpreted